In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from pathlib import Path
import json
load_dotenv()

True

In [2]:
DATA = Path('data')

standards = (DATA / 'team_standards.md').read_text("utf-8")
payment_code = (DATA / 'payment_utils.py').read_text("utf-8")
auth_code = (DATA / 'auth.py').read_text("utf-8")

In [3]:
def call_llm(prompt):
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [4]:
# print(standards)

prompt = f"""Review this Python code against our team standards.
Give 2-3 issues with line numbers. Do NOT claim you posted a PR comment.

STANDARDS:
{standards}

FILE : payment_utils.py

{payment_code}

"""

call_llm(prompt)

'Here are a few issues identified in the code against the team standards:\n\n1. **Line 6**: The code uses string interpolation to construct an SQL query, which can lead to SQL injection vulnerabilities. It should use parameterized queries with SQLAlchemy instead.\n   \n   ```python\n   query = f"SELECT * FROM users WHERE id = {user_id}"\n   ```\n\n2. **Line 9**: The `process_payment` function prints the card number, which violates the standard of never logging sensitive data (such as card numbers). This could lead to potential leakage of sensitive information.\n\n   ```python\n   print(f"Processing {amount} for card {card_number}")\n   ```\n\n3. **Line 2**: There is no validation of the `user_id` input before it is used in a DB query. The input should be validated to ensure it meets the expected format and constraints before performing any database operations.\n\n   ```python\n   def get_user(user_id):\n   ```\n\nThese issues need to be addressed to comply with the team\'s coding stand

In [5]:
print(call_llm(prompt))

Here are a few issues with the code in `payment_utils.py` based on the provided team standards:

1. **Line 6 - Use of Non-parameterized Query**: 
   - The query in `get_user(user_id)` is built using string interpolation, which can lead to SQL injection vulnerabilities. Instead, parameterized queries using SQLAlchemy should be used.

   ```python
   query = f"SELECT * FROM users WHERE id = {user_id}"
   ```

2. **Line 9 - Logging Sensitive Data**: 
   - In the `process_payment(amount, card_number)` function, the card number is printed to the console. Logging sensitive data such as credit card numbers is against team standards.

   ```python
   print(f"Processing {amount} for card {card_number}")
   ```

3. **Line 11 - Missing User Input Validation**: 
   - The function `get_user(user_id)` lacks validation on `user_id`. Before executing the database query, user input must be validated to ensure it adheres to expected formats or types.

These issues should be addressed to adhere to the te

In [6]:
##tools
#allowed_files = ['payment_utils.py', 'auth.py']

pr = {
    "number": 247,
    "comments": [],
    "status": "PENDING",
}


def read_file(filename):
    return (DATA / filename).read_text("utf-8")

def post_review_comment(filename,comment):
    pr['comments'].append({
        "filename": filename,
        "comment": comment
    })
    return "Comment Saved"

def set_pr_status(status):
    if status not in ["PASS", "BLOCKED"]:
        return "ERROR: status must be either PASS or BLOCKED"
    pr['status'] = status
    return f"PR status updated"



In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the content of a file. Use this to read files that are part of the PR.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string"
                    }
                },
                "required": ["filename"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "post_review_comment",
            "description": "Post a review comment on the PR. Use this to provide feedback on the code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string"
                    },
                    "comment": {
                        "type": "string"
                    }
                },
                "required": ["filename", "comment"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_pr_status",
            "description": "Set the status of the PR. Use this to indicate whether the PR is ready to be merged or not.",
            "parameters": {
                "type": "object",
                "properties": {
                    "status": {
                        "type": "string",
                        "enum": ["PASS", "BLOCKED"]
                    }
                },
                "required": ["status"]
            }
        }
    }
]


In [8]:
demo_prompt = [
    {"role": "user", "content": "Please read auth.py before reviewing it"}
]


client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_prompt,
    tools=tools
)

response.choices[0].message.content


In [9]:
print(response)

ChatCompletion(id='chatcmpl-EfTYoG49NlTgLDBoTho3cc7BrM319', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_aEgyk6QSTX8XyAaqoXXcooaC', function=Function(arguments='{"filename":"auth.py"}', name='read_file'), type='function')]))], created=1793553786, model='gpt-4o-mini-2024-07-18', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_fdb7cdcf55', usage=CompletionUsage(completion_tokens=15, prompt_tokens=143, total_tokens=158, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))


In [10]:
first_message = response.choices[0].message

In [11]:
#first_message.tool_calls[0].function.name
#first_message.tool_calls[0].function.arguments


In [12]:
call = first_message.tool_calls[0]

In [13]:
arguments = json.loads(first_message.tool_calls[0].function.arguments)

In [14]:
file_text = read_file(arguments['filename'])
file_text

'def login(username, password):\n    if username == "admin" and password == "admin123":\n        return True\n    return False\n\n\ndef generate_token(user_id):\n    import time\n    return f"{user_id}_{time.time()}"'

In [15]:
assistant_message = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": call.id,
            "type": "function",
            "function": {
                "name": call.function.name,
                "arguments": call.function.arguments,
            },
        }
    ],
}

demo_prompt.append(assistant_message)


In [16]:
demo_prompt

[{'role': 'user', 'content': 'Please read auth.py before reviewing it'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_aEgyk6QSTX8XyAaqoXXcooaC',
    'type': 'function',
    'function': {'name': 'read_file',
     'arguments': '{"filename":"auth.py"}'}}]}]

In [17]:
demo_prompt.append(
    {
        "role": "tool",
        "tool_call_id": call.id,
        "name": call.function.name,
        "content": file_text,
    }
)


In [18]:
response_se = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=demo_prompt,
    tools=tools
)

print(response_se.choices[0].message.content)


The `auth.py` file contains two functions:

1. **`login(username, password)`**:
   - This function checks if the provided `username` and `password` match the hardcoded credentials (`"admin"` and `"admin123"`).
   - If they match, it returns `True`; otherwise, it returns `False`.

2. **`generate_token(user_id)`**:
   - This function generates a token by concatenating the `user_id` with the current timestamp.
   - The token format is `"{user_id}_{timestamp}"`, which could potentially be used for session management or API authentication.

### Review Comments:
1. **Security Concerns**:
   - The `login` function uses hardcoded credentials which is not secure. This method should be replaced with a more secure authentication mechanism. Consider using hashed passwords and a database.
   - The `generate_token` function does not seem to use any cryptographic methods for token generation. Consider using a library like `secrets` or `jwt` for generating secure tokens.

2. **Code Readability**:
   -